# Microchannel (MOFF - Multi Orifice Fluid Fraction) with XNSE-Solver

In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 217
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 104
   at Submission#43.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.XdgTimestepping;

In [ ]:
BoSSSshell.WorkflowMgm.Init("MOFFMicroChannel");

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,[ ],\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


## Problem settings

In [ ]:
double density = 1000.0;
double viscosity = 1.0e-3;
double averBulkVelocity = 1.0;
double SizeScale = 1.0e-6;
double H = 50.0 * SizeScale;   // channel height
double We = 200.0 * SizeScale;
double Wi = 50.0 * SizeScale;
double Dh = 2 * Wi * H / (Wi + H);  // hydraulic diameter

double Re = density * Dh * averBulkVelocity / viscosity;
Re

49.99999999999999

In [ ]:
double Re_e = density * Wi * (3.0/2.0)*averBulkVelocity / viscosity;
Re_e

75

## Grid Creation

Construct a grid consisting of three parts:
- 1: Inlet

- 2.1: centerline expansion section
- 2.2: outer expansion section (lower)
- 2.3: outer expansion section (upper)
- 2.4: contraction section (upstream)
- 2.5: contraction section (downstream)

- 3: outlet

In [ ]:
double Le = 200.0 * SizeScale;
double Ls = 100.0 * SizeScale;
double L = Le + Ls;

In [ ]:
int FlowResolution = 1;

### inlet

In [ ]:
double Linlet = 200.0 * SizeScale;

In [ ]:
int InletStreamWiseResolution = 10 * FlowResolution;
int ContractionCrossWiseResolution = 5 * FlowResolution;
int ContractionStreamWiseResolution = 5 * FlowResolution;
int OuterExpansionCrossWiseResolution = 5 * FlowResolution;

In [ ]:
double[] xNodes1 = GenericBlas.Linspace(-Linlet -Ls/2.0, -Ls/2.0, InletStreamWiseResolution + 1);
double[] yNodes1 = GenericBlas.Linspace(-Wi/2.0, Wi/2.0, ContractionCrossWiseResolution + 1);
var grd1 = Grid2D.Cartesian2DGrid(xNodes1, yNodes1);

In [ ]:
double[] xNodes2 = xNodes1;
double[] yNodes2 = GenericBlas.Linspace(-We/2.0, -Wi/2.0, OuterExpansionCrossWiseResolution + 1);
var grd2 = Grid2D.Cartesian2DGrid(xNodes2, yNodes2);

In [ ]:
double[] xNodes3 = xNodes1;
double[] yNodes3 = GenericBlas.Linspace(Wi/2.0, We/2.0, OuterExpansionCrossWiseResolution + 1);
var grd3 = Grid2D.Cartesian2DGrid(xNodes3, yNodes3);

In [ ]:
double[] xNodes5 = GenericBlas.Linspace(-Ls/2.0, 0.0, ContractionStreamWiseResolution + 1);
double[] yNodes5 = yNodes1;
var grd4 = Grid2D.Cartesian2DGrid(xNodes5, yNodes5);

In [ ]:
var gridInlet = GridCommons.MergeLogically(new GridCommons[] {grd1, grd2, grd3, grd4});
//var grd = GridCommons.Seal(gridInlet, 4);

In [ ]:
Formula ExpansionInletVelocity = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " +
    "double WeHalf = 100.0e-6;" +
    "double U_max = (3.0/2.0)*0.25;" + 
    "return (-U_max / WeHalf.Pow2()) * X[1].Pow2() + U_max; } "
);

### repetive sections

In [ ]:
double Le = 200.0 * SizeScale;
double L = Le + Ls;

In [ ]:
int ExpansionStreamWiseResolution = 20 * FlowResolution;

In [ ]:
int nSections = 1;
double xOffset = L/2.0;

In [ ]:
double[] xNodes1 = GenericBlas.Linspace(xOffset - Le/2.0, xOffset + Le/2.0, ExpansionStreamWiseResolution + 1);
double[] yNodes1 = GenericBlas.Linspace(-Wi/2.0, Wi/2.0, ContractionCrossWiseResolution + 1);
var grd1 = Grid2D.Cartesian2DGrid(xNodes1, yNodes1);

In [ ]:
double[] xNodes2 = xNodes1;
double[] yNodes2 = GenericBlas.Linspace(-We/2.0, -Wi/2.0, OuterExpansionCrossWiseResolution + 1);
var grd2 = Grid2D.Cartesian2DGrid(xNodes2, yNodes2);

In [ ]:
double[] xNodes3 = xNodes1;
double[] yNodes3 = GenericBlas.Linspace(Wi/2.0, We/2.0, OuterExpansionCrossWiseResolution + 1);
var grd3 = Grid2D.Cartesian2DGrid(xNodes3, yNodes3);

In [ ]:
double[] xNodes4 = GenericBlas.Linspace(xOffset - L/2.0, xOffset - Le/2.0, ContractionStreamWiseResolution + 1);
double[] yNodes4 = yNodes1;
var grd4 = Grid2D.Cartesian2DGrid(xNodes4, yNodes4);

In [ ]:
double[] xNodes5 = GenericBlas.Linspace(xOffset + Le/2.0, xOffset + L/2.0, ContractionStreamWiseResolution + 1);
double[] yNodes5 = yNodes1;
var grd5 = Grid2D.Cartesian2DGrid(xNodes5, yNodes5);

In [ ]:
var gridSections = GridCommons.MergeLogically(new GridCommons[] {grd1, grd2, grd3, grd4, grd5});
//var grd = GridCommons.Seal(gridSection, 4);

### outlet

In [ ]:
double Loutlet = 400.0 * SizeScale;
double Lsections = nSections * L;
Lsections

0.0003

In [ ]:
double Lend = Lsections + Ls/2.0 + Loutlet;
Lend

0.0007499999999999999

In [ ]:
int OutletStreamWiseResolution = 20 * FlowResolution;

In [ ]:
double[] xNodes1 = GenericBlas.Linspace(Lsections, Lsections + Ls/2.0, ContractionStreamWiseResolution + 1);
double[] yNodes1 = GenericBlas.Linspace(-Wi/2.0, Wi/2.0, ContractionCrossWiseResolution + 1);
var grd1 = Grid2D.Cartesian2DGrid(xNodes1, yNodes1);

In [ ]:
double[] xNodes2 = GenericBlas.Linspace(Lsections + Ls/2.0, Lend, OutletStreamWiseResolution + 1);
double[] yNodes2 = GenericBlas.Linspace(-Wi/2.0, Wi/2.0, ContractionCrossWiseResolution + 1);
var grd2 = Grid2D.Cartesian2DGrid(xNodes2, yNodes2);

In [ ]:
double[] xNodes3 = xNodes2;
double[] yNodes3 = GenericBlas.Linspace(-We/2.0, -Wi/2.0, OuterExpansionCrossWiseResolution + 1);
var grd3 = Grid2D.Cartesian2DGrid(xNodes3, yNodes3);

In [ ]:
double[] xNodes4 = xNodes2;
double[] yNodes4 = GenericBlas.Linspace(Wi/2.0, We/2.0, OuterExpansionCrossWiseResolution + 1);
var grd4 = Grid2D.Cartesian2DGrid(xNodes4, yNodes4);

In [ ]:
var gridOutlet = GridCommons.MergeLogically(new GridCommons[] {grd1, grd2, grd3, grd4});
//var grd = GridCommons.Seal(gridInlet, 4);

### System

In [ ]:
var gridSystem = GridCommons.MergeLogically(new GridCommons[] {gridInlet, gridSections, gridOutlet});
var grd = GridCommons.Seal(gridSystem, 4);

In [ ]:
grd.DefineEdgeTags(delegate(Vector X) {
    string ret = null;
    
    if ((X.y + We/2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.y - We/2.0).Abs() <= 1e-8)
        ret = IncompressibleBcType.Wall.ToString();

    // inlet
    if ((X.x + (Linlet + Ls/2.0)).Abs() <= 1e-8)
        ret = IncompressibleBcType.Velocity_Inlet.ToString();
    // if ((X.x).Abs() <= 1e-8)
    //     ret = IncompressibleBcType.Pressure_Outlet.ToString();
    if ((X.x + Ls/2.0).Abs() <= 1e-8 && (X.y.Abs() - Wi/2.0) > 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.y.Abs() - Wi/2.0).Abs() <= 1e-8 && (X.x > -Ls/2.0) && (X.x < 0.0))
        ret = IncompressibleBcType.Wall.ToString();

    // repetitive Section
    // if ((X.x + L/2.0).Abs() <= 1e-8)
    //     ret = IncompressibleBcType.Velocity_Inlet.ToString();
    // if ((X.x - Lend).Abs() <= 1e-8)
    //     ret = IncompressibleBcType.Pressure_Outlet.ToString();
    if ((X.x - Ls/2.0).Abs() <= 1e-8 && (X.y.Abs() - Wi/2.0) > 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.x - (Ls/2.0 + Le)).Abs() <= 1e-8 && (X.y.Abs() - Wi/2.0) > 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.y.Abs() - Wi/2.0).Abs() <= 1e-8 && (X.x > 0.0) && (X.x < Ls/2.0))
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.y.Abs() - Wi/2.0).Abs() <= 1e-8 && (X.x > Le + Ls/2.0) && (X.x < Le + Ls))
        ret = IncompressibleBcType.Wall.ToString();

    // outlet
    if ((X.x - Lend).Abs() <= 1e-8)
        ret = IncompressibleBcType.Pressure_Outlet.ToString();
    if ((X.x - (Lsections + Ls/2.0)).Abs() <= 1e-8 && (X.y.Abs() - Wi/2.0) > 1e-8)
        ret = IncompressibleBcType.Wall.ToString();
    if ((X.y.Abs() - Wi/2.0).Abs() <= 1e-8 && (X.x > Lsections) && (X.x < Lsections + Ls/2.0))
        ret = IncompressibleBcType.Wall.ToString();


    return ret;
}); 

In [ ]:
grd.NumberOfCells

850

## Set up Controls 

In [ ]:
int k = 2;

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
XNSE_Control C = new XNSE_Control();

C.SetGrid(grd);
C.SetDGdegree(k);

C.PhysicalParameters.rho_A = density;
C.PhysicalParameters.mu_A = viscosity;
C.PhysicalParameters.IncludeConvection = true;

C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX#A", ExpansionInletVelocity);

C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.SessionName = "MOFF_SetupTestSystem2";

Controls.Add(C);

## Launch Jobs

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,MOFF_SetupTestSystem2


In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();

    oneJob.NumberOfMPIProcs = 1;

    int numThreads = 2;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    oneJob.Activate(myBatch); 
}

## Postprocessing

In [ ]:
wmg.Sessions

#0: MOFFMicroChannel	MOFF_SetupTestSystem2	01/08/2025 14:24:09	89c8ef1e...
#1: MOFFMicroChannel	MOFF_SetupTestSystem	01/08/2025 14:15:14	b1ba9a3b...
#2: MOFFMicroChannel	MOFF_SetupTestInletSection	01/08/2025 13:53:36	a518f51a...
#3: MOFFMicroChannel	MOFF_SetupTestInlet*	01/08/2025 13:19:45	4ba2e311...
#4: MOFFMicroChannel	MOFF_SetupTest	01/08/2025 12:33:31	73f93374...


In [ ]:
var sess = wmg.Sessions.Pick(0);
sess

MOFFMicroChannel	MOFF_SetupTestSystem2	01/08/2025 14:24:09	89c8ef1e...

In [ ]:
sess.Export().WithSupersampling(2).Do();